# Transaction Foundation Model on Ray — Part 4: Pretrain with Ray Train

<div align="left">
  <a target="_blank" href="https://console.anyscale.com/template-preview/fintech_transaction_fm"><img src="https://img.shields.io/badge/🚀 Run_on-Anyscale-9hf"></a>&nbsp;
  <a href="https://github.com/anyscale/templates/tree/main/templates/fintech_transaction_fm" role="button"><img src="https://img.shields.io/static/v1?label=&message=View%20On%20GitHub&color=586069&logo=github&labelColor=2f363d"></a>&nbsp;
</div>

**⏱️ Time to complete**: ~5 min

---

In Part 3 we turned each card's history into token windows. Here we pretrain the foundation model on those windows by masked-feature modeling, using Ray Train to run a plain-PyTorch training loop across the cluster.

In [ ]:
import sys, os, json

DEMO_ROOT = os.path.abspath(os.getcwd())
if DEMO_ROOT not in sys.path:
    sys.path.insert(0, DEMO_ROOT)

import pandas as pd
import logging
import ray
ray.init(ignore_reinit_error=True, runtime_env={"working_dir": DEMO_ROOT},   logging_level=logging.ERROR))

from src.paths import artifact_paths, get_demo_base_dir
from src.scale_config import load_scale

SCALE = "full"                       # same knob as the earlier parts; mini = tiny, CPU-only
cfg = load_scale(SCALE)
BASE = get_demo_base_dir()
paths = artifact_paths(BASE, SCALE)

SyntaxError: unmatched ')' (121974697.py, line 10)

(autoscaler +6h40m20s) [autoscaler] [8CPU-32GB] Attempting to add 1 node to the cluster (increasing from 3 to 4).
(autoscaler +6h40m20s) [autoscaler] [8CPU-32GB|m5.2xlarge] [us-west-2c] [on-demand] Launched 1 instance.


## Pretrain by predicting masked fields

We pretrain the model the way BERT pretrains on text: mask a fraction (15%) of the dynamic-field tokens across a window and have the model predict the originals from the surrounding transactions. There are no labels involved — the transactions supervise themselves, which is what lets a foundation model learn from far more data than the scarce fraud labels alone.

The encoder is **bidirectional** because the tasks this model feeds (fraud, churn, credit) are fixed-window classification, where seeing both sides of a position beats left-to-right next-token prediction. Each dynamic field gets its own prediction head, and because the heads differ wildly in difficulty — `day_of_week` is 10-way, `merchant` is ~2,000-way — they're balanced with learned **uncertainty weighting** (Kendall & Gal) so the hard head doesn't dominate the gradient. The model and the loss live in `src/model.py`; they're deliberately small, because the model is not the hard part here — the engineering around it is.

## The training loop is plain PyTorch — Ray Train scales it

The function that does the work (`train_func` in `src/pretrain.py`) is ordinary PyTorch: iterate batches, mask, forward, backward, step. Ray Train wraps it with the distributed machinery you'd otherwise hand-write — it launches the workers, shards the dataset across them (`get_dataset_shard` + `iter_torch_batches`), wraps the model in DDP (`prepare_model`), and persists checkpoints to shared storage.

The payoff is the `ScalingConfig`: moving from one CPU worker here to many GPU workers at `full` is a one-line change (`num_workers`, `use_gpu`), not a rewrite of the loop. `scripts/03_pretrain.py` wraps this same `train_func` for headless runs, so the walkthrough and the job train identically.

Note: this takes about 12 min on 'small' w/ 1x L40s

In [ ]:
from ray.train import ScalingConfig, RunConfig
from ray.train.torch import TorchTrainer
from src.pretrain import train_func, save_checkpoint   # train_func is shared with scripts/03_pretrain.py

pt = cfg["pretrain"]
seq_len = cfg["tokenize"]["seq_len"]
train_ds = ray.data.read_parquet(paths["tokenized_pretrain"])

trainer = TorchTrainer(
    train_func,                                       # the per-worker PyTorch loop (src/pretrain.py)
    train_loop_config={
        "vocab_path": paths["vocab"], "arch": cfg["model"], "size": SCALE, "max_len": seq_len,
        "epochs": pt["epochs"], "batch_size": pt["batch_size"], "lr": pt["lr"],
        "use_fsdp": pt["use_fsdp"], "loss_weighting": "uncertainty", "seed": 0,
    },
    # The one line that moves laptop -> cluster: number of workers, CPU vs GPU.
    scaling_config=ScalingConfig(num_workers=pt["num_workers"], use_gpu=pt["use_gpu"]),
    datasets={"train": train_ds},                     # Ray Train shards this across the workers
    run_config=RunConfig(
        name="transaction_fm_pretrain",
        storage_path=os.path.join(BASE, "ray_results"),  # shared storage: workers run on other nodes
    ),
)
result = trainer.fit()
save_checkpoint(result, paths["checkpoint"])           # copy weights to the canonical path for Part 5
print(f"done — final mlm_loss {result.metrics['mlm_loss']:.2f}, "
      f"macro accuracy {result.metrics['acc_macro']:.3f} -> {paths['checkpoint']}")

(TrainController pid=58215) A run snapshot was found in storage folder at: '/mnt/cluster_storage/transaction-fm/ray_results/transaction_fm_pretrain'
(TrainController pid=58215) This snapshot contains a list of checkpoints reported via `ray.train.report` and will be loaded. This allows the latest checkpoint found in the snapshot to be accessible within your training function via `ray.train.get_checkpoint`.
(TrainController pid=58215) If you meant to start a brand new training job without any information about previous checkpoints found in this directory, please configure a new, unique `RunConfig(name)` or delete the existing folder at '/mnt/cluster_storage/transaction-fm/ray_results/transaction_fm_pretrain'.
(TrainController pid=58215) Requesting resources: {'GPU': 1} * 2
(TrainController pid=58215) [State Transition] INITIALIZING -> SCHEDULING.
(TrainController pid=58215) Attempting to start training worker group of size 2 with the following resources: [{'GPU': 1}] * 2


(autoscaler +15s) Tip: use `ray status` to view detailed cluster status. To disable these messages, set RAY_SCHEDULER_EVENTS=0.
(autoscaler +15s) [autoscaler] [4xL40S:48CPU-384GB] Attempting to add 1 node to the cluster (increasing from 0 to 1).
(autoscaler +15s) [autoscaler] [4xL40S:48CPU-384GB|g6e.12xlarge] [us-west-2c] [on-demand] Launched 1 instance.


(TrainController pid=58215) [FailurePolicy] RETRY
(TrainController pid=58215)   Source: controller
(TrainController pid=58215)   Error count: 1 (max allowed: inf)
(TrainController pid=58215) Error: Training failed due to controller error:
(TrainController pid=58215) The worker group startup timed out after 60.0 seconds waiting for 2 workers. Potential causes include: (1) temporary insufficient cluster resources while waiting for autoscaling (ignore this warning in this case), (2) infeasible resource request where the provided `ScalingConfig` cannot be satisfied), and (3) transient network issues. Set the RAY_TRAIN_WORKER_GROUP_START_TIMEOUT_S environment variable to increase the timeout.
(TrainController pid=58215) Traceback (most recent call last):
(TrainController pid=58215)   File "/home/ray/anaconda3/lib/python3.12/site-packages/ray/train/v2/_internal/execution/controller/controller.py", line 414, in _start_worker_group
(TrainController pid=58215)     self._worker_group = self.work

(autoscaler +1m25s) [autoscaler] Cluster upscaled to {56 CPU, 4 GPU}.


(RayTrainWorker pid=3975, ip=10.0.129.254) Setting up process group for: env:// [rank=0, world_size=2]
(TrainController pid=58215) Started training worker group of size 2: 
(TrainController pid=58215) - (ip=10.0.129.254, pid=3975) world_rank=0, local_rank=0, node_rank=0
(TrainController pid=58215) - (ip=10.0.129.254, pid=3976) world_rank=1, local_rank=1, node_rank=0
(TrainController pid=58215) [State Transition] SCHEDULING -> RUNNING.
(RayTrainWorker pid=3975, ip=10.0.129.254) /tmp/ray/session_2026-06-30_13-35-30_925017_2803/runtime_resources/working_dir_files/_ray_pkg_cc8016085f2efc85/src/model.py:79: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
(RayTrainWorker pid=3975, ip=10.0.129.254)   self.encoder = nn.TransformerEncoder(layer, num_layers=n_layers)
(RayTrainWorker pid=3975, ip=10.0.129.254) Moving model to device: cuda:0
(RayTrainWorker pid=3975, ip=10.0.129.254) Wrapping provided model in DistributedData

...

(pid=59307) - ListFiles:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=59307) - ReadFiles:   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(pid=59307) - split(2, equal=True):   0%|          | 0.00/1.00 [00:00<?, ? row/s]

(RayTrainWorker pid=3975, ip=10.0.129.254) Reporting training result 32: TrainingReport(checkpoint=None, metrics={'epoch': 13, 'mlm_loss': 7.42941739378857, 'acc_amount_bucket': 0.4547536979217115, 'ppl_amount_bucket': 4.044979322809036, 'acc_merchant_bucket': 0.18966342117687196, 'ppl_merchant_bucket': 40.823086835755866, 'acc_merchant_category': 0.3851273729177371, 'ppl_merchant_category': 5.049960523498698, 'acc_mcc': 0.2791218274288134, 'ppl_mcc': 11.81290262508193, 'acc_hour': 0.34719512181240725, 'ppl_hour': 8.660711860240063, 'acc_day_of_week': 0.15787165723887367, 'ppl_day_of_week': 6.968219595496247, 'acc_macro': 0.30228884974940246}, validation=False)
(SplitCoordinator pid=59307) ✔️  Dataset train_8_14 execution finished in 32.18 seconds
(SplitCoordinator pid=59307) INFO:openlineage.client.client:OpenLineageClient will use `composite` transport
(SplitCoordinator pid=59307) INFO:openlineage.client.transport.composite:Stopping OpenLineage CompositeTransport emission after the f

done — final mlm_loss 7.40, macro accuracy 0.303 -> /mnt/cluster_storage/transaction-fm/model/small/


## Reading the metrics

Watch the per-field numbers, not the weighted total — the total drifts as the uncertainty weights learn. For each dynamic field we report top-1 accuracy and perplexity. The tell that the model is learning structure rather than guessing is **perplexity well below the field's vocab size**: a head that learned nothing sits around the vocab size.

At `mini` (2 CPU epochs, a 2-layer model) this is only a smoke test, but even here most fields clear that bar — amount lands around perplexity 6 against 19 buckets, `merchant_category` around 7 against 12. The real signal, and the downstream fraud lift, comes from the `small`/`full` GPU pretrain.

In [ ]:
from src.tokenizer import DYNAMIC_FIELDS, field_vocab_sizes

m = result.metrics
sizes = field_vocab_sizes()
print(f"final MLM loss {m['mlm_loss']:.2f}   ·   macro accuracy {m['acc_macro']:.3f}\n")
tbl = pd.DataFrame([
    {"field": f, "accuracy": round(m[f"acc_{f}"], 3),
     "perplexity": round(m[f"ppl_{f}"], 1), "vocab": sizes[f]}
    for f in DYNAMIC_FIELDS
])
print(tbl.to_string(index=False))

final MLM loss 7.40   ·   macro accuracy 0.303

            field  accuracy  perplexity  vocab
    amount_bucket     0.454         4.0     19
  merchant_bucket     0.191        40.5   2003
merchant_category     0.387         5.0     12
              mcc     0.280        11.8    131
             hour     0.347         8.7     27
      day_of_week     0.161         6.9     10


(autoscaler +13m40s) [autoscaler] Downscaling node i-0630eb8ac173f97b9 (node IP: 10.0.154.114) due to node idle termination.
(autoscaler +13m40s) [autoscaler] Cluster resized to {112 CPU, 4 GPU}.
(autoscaler +16m40s) [autoscaler] Downscaling node i-0a37904a0338e86ad (node IP: 10.0.152.214) due to node idle termination.
(autoscaler +16m40s) [autoscaler] Cluster resized to {48 CPU, 4 GPU}.
(autoscaler +16m45s) [autoscaler] Downscaling node i-0576c9c257b3fd6e5 (node IP: 10.0.129.254) due to node idle termination.


## Takeaways

**Ray Train**
- The training loop is plain PyTorch; Ray Train adds the distributed parts — worker launch, dataset sharding, DDP, checkpointing, fault tolerance — without changing the loop.
- `ScalingConfig` is the one knob that moves the same code from one CPU worker here to N GPU workers at `full` (`num_workers`, `use_gpu`); the model fits one GPU at every scale, so it's data-parallel DDP, with `use_fsdp` available if the model ever outgrows a GPU.
- Checkpoints persist to shared storage (`storage_path`), so workers on other nodes can write them and downstream stages can read them.
- The notebook runs the same `train_func` that `scripts/03_pretrain.py` runs headless.

**The model**
- Pretraining is self-supervised masked-feature modeling: predict masked dynamic-field tokens from bidirectional context — no labels required.
- One prediction head per dynamic field, balanced by learned uncertainty weighting so the ~2,000-way merchant head doesn't swamp the 10-way day-of-week head.
- Read per-field perplexity against vocab size to confirm it's learning; the trained encoder becomes the customer embedding in Part 5.

---

## Next

**Part 5 — Batch embedding extraction**: run the pretrained encoder over every card with Ray Data — a heterogeneous CPU-read + GPU-infer pipeline that writes one embedding per window.